# Dofus Brain — Domain-Native AI training on a Colab T4

This notebook is the **GPU-only** path for the Dofus AI. It trains
`DofusBrainTransformer` from scratch on the domain corpus (combat policy +
trilingual intent), runs the certification benchmark, optionally distils into
the TRM-ACT combat reasoner, and packages everything for download.

## Open this in VS Code with the Colab extension

1. Install the **Colab** extension in VS Code (Google's official one).
2. Open this file, then pick a runtime: *Select Kernel -> Colab -> T4 GPU*.
3. Run the cells top to bottom.

## What it produces

| Artifact | What it is |
| --- | --- |
| `dofus_brain.pt` | Teacher transformer weights (multi-task: stance, intent, value, MLM) |
| `vocab.json` | Domain tokenizer vocabulary, built from **training text only** |
| `config.json` | Architecture + label maps + best epoch/loss |
| `training_report.json` | Per-epoch history, confusion matrices, per-class F1 |
| `benchmark_report.json` | Certification battery, held-out generalisation, adversarial safety |
| `trm_act_combat.pt` | Distilled sub-millisecond student (only if the gate passes) |
| `dofus-brain-native-colab.zip` | Everything above, one download |

## What changed from the naive loop

- **Leak-free split.** Samples are grouped by normalised text; whole groups are
  held out, and the vocabulary is built from training text only.
- **Real validation.** Every epoch reports held-out loss/accuracy and the best
  epoch is checkpointed, not the last one.
- **Class-weighted intent loss.** `NONE` dominates the corpus; the loss is
  re-weighted and the report carries a per-class breakdown.
- **Bounded value targets.** The value head predicts a [0,1] reward proxy.
- **Benchmark gate.** Distillation refuses to run unless the teacher certifies.

## Honest note on scale

The generated corpus is on the order of tens of thousands of synthetic samples.
It teaches **format, intent routing and stance selection**, not encyclopedic
game knowledge. The held-out generalisation battery exists precisely so a
memorising model cannot look good: read that number, not the aggregate.


## 1. Environment

Confirm the GPU and PyTorch build before spending quota.

In [ ]:
import platform, sys
print('python', sys.version.split()[0], '|', platform.platform())
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('gpu:', torch.cuda.get_device_name(0))
    print('sm count:', props.multi_processor_count,
          '| total mem GB:', round(props.total_memory / 2**30, 2))
    print('bfloat16 supported:', props.major >= 8)
    print('=> T4 (sm75) has FP16 Tensor Cores but NO bfloat16.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('training device:', DEVICE)

## 2. Get the repository

Clone into `/content` (local VM disk), not Drive: Drive I/O would bottleneck
the data loader.

In [ ]:
import shutil, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/saoxdxd2/dofus-emu.git'
CLONE_DIR = Path('/content/dofus-emu')
REPO_DIR = CLONE_DIR / 'dofus-network'

if CLONE_DIR.exists():
    shutil.rmtree(CLONE_DIR)

result = subprocess.run(
    ['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_DIR)],
    capture_output=True, text=True)
print(result.stdout.strip() or result.stderr.strip())
print('clone exit code:', result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        'git clone failed. If the repo is private, use a token in the URL:\n'
        '  https://<token>@github.com/saoxdxd2/dofus-emu.git')

for required in ('ai/train_from_scratch.py', 'ai/dofus_native_model.py',
                 'ai/dofus_knowledge_benchmark.py', 'ai/pipeline.py',
                 'data/generate_expanded_corpus.py',
                 'data/generate_combat_policy_dataset.py'):
    print(f'  {required:48s}', (REPO_DIR / required).exists())
if not (REPO_DIR / 'ai' / 'train_from_scratch.py').exists():
    raise FileNotFoundError(
        f'Clone succeeded but the ai/ package is missing.\n'
        f'Contents: {sorted(p.name for p in CLONE_DIR.iterdir())}')

In [ ]:
import importlib, sys

# Drop stale sys.path entries so an old clone cannot shadow this one.
sys.path[:] = [p for p in sys.path if 'dofus-network' not in p and 'dofus-emu' not in p]
sys.path.insert(0, str(REPO_DIR))
importlib.invalidate_caches()

from ai.train_from_scratch import INTENT_MAP, STANCE_MAP, STANCE_VALUE
from ai.dofus_native_model import DofusBrainTransformer, DofusDomainTokenizer
print('intent classes :', len(INTENT_MAP), list(INTENT_MAP))
print('stance classes :', len(STANCE_MAP), list(STANCE_MAP))
print('stance values  :', STANCE_VALUE)

## 3. Generate the domain corpus (deterministic)

Seeded generation means two runs with the same `SEED` produce byte-identical
data, so a split is reproducible and a metric is traceable.

In [ ]:
import json, subprocess, sys
from pathlib import Path

SEED = 42
COMBAT_COUNT = 20000   # tactical combat scenarios (4 balanced stances)
FORCE_REGENERATE = True

def run(step, args):
    print('=' * 70, flush=True)
    print('STEP:', step, flush=True)
    print('=' * 70, flush=True)
    cmd = [sys.executable, '-u', '-m', *args]
    print('$', ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=str(REPO_DIR))
    if result.returncode != 0:
        raise RuntimeError(f'{step} failed with exit code {result.returncode}')

combat_file = REPO_DIR / 'data' / 'dofus_combat_policy_dataset.json'
corpus_file = REPO_DIR / 'data' / 'dofus_deep_training_corpus.json'

if FORCE_REGENERATE or not combat_file.exists():
    run('combat scenarios', ['data.generate_combat_policy_dataset',
                             '--count', str(COMBAT_COUNT), '--seed', str(SEED)])
if FORCE_REGENERATE or not corpus_file.exists():
    run('expanded corpus', ['data.generate_expanded_corpus', '--seed', str(SEED)])

corpus = json.loads(corpus_file.read_text(encoding='utf-8'))
counts = {}
for s in corpus:
    counts[s.get('type', 'UNKNOWN')] = counts.get(s.get('type', 'UNKNOWN'), 0) + 1
print('\ncorpus samples:', len(corpus))
print('by type       :', counts)
distinct = len({s.get('text', '') for s in corpus})
print('distinct texts:', distinct)
print('note: duplicate texts collapse into one split group, so distinct texts')
print('      is the number that actually matters for the validation split.')

## 4. Inspect the split before training

Confirm the leak-free guarantee holds *before* the GPU bill: zero shared text
groups between train and validation.

In [ ]:
from collections import Counter
from ai.train_from_scratch import split_samples

VALID_FRACTION = 0.15
train_samples, valid_samples, shared = split_samples(
    corpus, valid_fraction=VALID_FRACTION, seed=SEED)

print(f'train samples      : {len(train_samples):,}')
print(f'valid samples      : {len(valid_samples):,}')
print(f'shared text groups : {shared}   <-- must be 0')
assert shared == 0, 'Split leaked text groups across train/valid'

intent_counts = Counter(
    s.get('label', 'NONE') for s in train_samples if s.get('type') != 'TACTICAL_COMBAT')
stance_counts = Counter(
    s.get('stance') or s.get('label') for s in train_samples if s.get('type') == 'TACTICAL_COMBAT')
print('\ntrain intent distribution :', dict(intent_counts.most_common()))
print('train stance distribution :', dict(stance_counts.most_common()))

## 5. Train the domain transformer

Trains `DofusBrainTransformer` (~12M params) from scratch. The trainer:

- builds the vocabulary from **training text only**,
- reports held-out loss / intent accuracy / stance accuracy / value MAE each epoch,
- early-stops on validation loss and restores the **best** epoch,
- writes `training_report.json` with confusion matrices and per-class F1.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

from ai.train_from_scratch import train_dofus_ai_from_scratch

EPOCHS = 15
BATCH_SIZE = 64
LR = 1e-3
PATIENCE = 5

OUTPUT_DIR = REPO_DIR / 'models' / 'dofus-brain-native'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model, tokenizer, report = train_dofus_ai_from_scratch(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    device=DEVICE,
    valid_fraction=VALID_FRACTION,
    seed=SEED,
    patience=PATIENCE,
    corpus_file=corpus_file,
    output_dir=OUTPUT_DIR,
    save=True,
)

print('\nbest epoch        :', report.best_epoch)
print('best valid loss   :', round(report.best_valid_loss, 5))
print('shared text groups:', report.shared_text_groups)
print('wall clock (s)    :', report.wall_clock_seconds)

## 6. Training diagnostics

Read the per-epoch curve and the per-class breakdown. Aggregate accuracy hides
a class the model never gets right; the per-class F1 does not.

In [ ]:
import json
from pathlib import Path

print(f"{'epoch':>5} {'train':>9} {'valid':>9} {'intent':>8} {'stance':>8} {'valMAE':>8}")
for row in report.history:
    print(f"{row['epoch']:>5} {row['train_loss']:>9.4f} {row['valid_loss']:>9.4f} "
          f"{row['valid_intent_accuracy']*100:>7.1f}% {row['valid_stance_accuracy']*100:>7.1f}% "
          f"{row['valid_value_mae']:>8.4f}")

print('\nPer-class intent precision/recall/F1 (held-out):')
print(f"{'class':<20} {'support':>8} {'prec':>7} {'recall':>7} {'f1':>7}")
for name, stats in sorted(report.per_class_intent.items(), key=lambda kv: -kv[1]['support']):
    print(f"{name:<20} {stats['support']:>8} {stats['precision']:>7.3f} "
          f"{stats['recall']:>7.3f} {stats['f1']:>7.3f}")

## 7. Certification benchmark

Three batteries matter here:

- **Domain batteries** — combat stances, zaaps/coords, resources, quests, ops.
- **Held-out generalisation** — paraphrases deliberately absent from the corpus.
  This is the honest read on whether the model generalises or memorises.
- **Adversarial safety** — empty strings and moderation notices must never map
  onto a destructive command such as `STOP` or `PASS_TURN`.

Distillation is gated on this passing.

In [ ]:
from ai.dofus_knowledge_benchmark import DofusKnowledgeBenchmark

THRESHOLD = 90.0
bench = DofusKnowledgeBenchmark(min_passing_score=THRESHOLD)
bench_result = bench.evaluate(model, tokenizer, device=torch.device(DEVICE))
bench_result.print_summary()

(OUTPUT_DIR / 'benchmark_report.json').write_text(json.dumps({
    'aggregate_score': bench_result.aggregate_score,
    'is_certified': bench_result.is_certified,
    'domain_scores': bench_result.domain_scores,
    'generalisation_score': bench_result.generalisation_score,
    'generalisation_passed': bench_result.generalisation_passed,
    'generalisation_tests': bench_result.generalisation_tests,
    'total_tests': bench_result.total_tests,
    'passed_tests': bench_result.passed_tests,
    'per_class': bench_result.per_class,
    'failure_reasons': bench_result.failure_reasons[:30],
}, indent=2), encoding='utf-8')
print('wrote', OUTPUT_DIR / 'benchmark_report.json')

## 8. Distil into TRM-ACT (gated)

Only runs when the benchmark certified the teacher. The student is the
sub-millisecond `TinyRecursiveReasoner` that the sentinel serves at runtime.

In [ ]:
import sys as _sys
ALLOW_UNCERTIFIED = False

if bench_result.is_certified or ALLOW_UNCERTIFIED:
    rc = subprocess.run(
        [_sys.executable, '-u', '-m', 'ai.distill',
         '--checkpoint', str(OUTPUT_DIR / 'dofus_brain.pt'),
         '--steps', '200', '--batch-limit', '512'],
        cwd=str(REPO_DIR)).returncode
    print('distillation exit code:', rc)
else:
    print('Teacher NOT certified -> distillation skipped.')
    print('This is the gate working as intended; see failure_reasons above.')

## 9. Package the artifacts

One zip with the weights, tokenizer, config, and every report.

In [ ]:
import shutil
from pathlib import Path

ARTIFACTS = ['dofus_brain.pt', 'vocab.json', 'config.json',
             'training_report.json', 'benchmark_report.json',
             'distillation_report.json', 'trm_act_combat.pt']

stage = Path('/content/dofus-brain-native-colab')
if stage.exists():
    shutil.rmtree(stage)
stage.mkdir(parents=True)

for name in ARTIFACTS:
    src = OUTPUT_DIR / name
    if src.exists():
        shutil.copy2(src, stage / name)
        print(f'  packaged {name:28s} {round(src.stat().st_size/2**20, 3)} MB')
    else:
        print(f'  skipped  {name:28s} (not produced)')

zip_path = Path(shutil.make_archive(
    '/content/dofus-brain-native-colab', 'zip', root_dir=stage.parent,
    base_dir=stage.name))
print('\nzip:', zip_path, round(zip_path.stat().st_size / 2**20, 3), 'MB')

## 10. Download

Downloads the zip plus the two files the runtime actually loads
(`dofus_brain.pt`, `vocab.json`). Also copies to Drive as a fallback.

In [ ]:
from google.colab import files  # type: ignore

files.download(str(zip_path))
for name in ('dofus_brain.pt', 'vocab.json', 'config.json'):
    path = OUTPUT_DIR / name
    if path.exists():
        files.download(str(path))

In [ ]:
# Drive fallback: persist artifacts even if the browser download fails.
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    dest = Path('/content/drive/MyDrive/dofus-brain-native')
    dest.mkdir(parents=True, exist_ok=True)
    for item in stage.iterdir():
        shutil.copy2(item, dest / item.name)
    shutil.copy2(zip_path, dest / zip_path.name)
    print('copied to', dest)
except Exception as exc:
    print('Drive copy skipped:', exc)

## 11. Where the files go on your machine

Unzip into the repo so the runtime finds them:

```
dofus-network/models/dofus-brain-native/
    dofus_brain.pt          <- teacher weights (served by the sentinel)
    vocab.json              <- domain tokenizer
    config.json             <- architecture + label maps
    training_report.json
    benchmark_report.json
    trm_act_combat.pt       <- distilled student (if certified)
```

The sentinel loads `models/dofus-thx01/` for the ModernBERT decision head and
`models/dofus-brain-native/trm_act_combat.pt` for the combat reasoner, so drop
the distilled student there to activate it.

Verify locally without a GPU:

```bash
python -m ai.pipeline --benchmark-only --checkpoint models/dofus-brain-native/dofus_brain.pt
```
